# Nettoyage de la base d'apprentissage

Maintenant que nous avons créé notre base d'apprentissage, il faut maintenant la nettoyer afin qu'elle soit pleinement utilisable pour nos futurs modèles qui prédiront la valeur marchande des joueurs de football.

## Partie A : Import de fonctions utiles au nettoyage de la base d'apprentissage

In [1]:
# Importation des packages nécessaires

import numpy as np
import pandas as pd
import sys
import os
import dill as pickle

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

In [2]:
# On charge la base d'apprentissage et celle des classements FIFA
df_base = pd.read_csv("../data_finale/base_apprentissage.csv")
df_fifa = pd.read_csv("../data/classement_fifa/fifa_ranking_fin_saison.csv", sep=",", encoding="utf-8-sig")

# On définit le dossier de sortie pour les fichiers nettoyés
dossier_sortie = r"..\data_finale"

## Partie B : Le netotyage de la base d'apprentissage

### B1 : Le traitement des doublons

Cette étape permet d'assainir et de consolider la base de données en deux temps :

**Diagnostic préliminaire** :
   * **Les doublons techniques :** Erreurs d'extraction ou de jointure (même joueur, même saison, même club).
   * **Les doublons de mercato :** Événements réels liés aux transferts de mi-saison (même joueur, même saison, mais plusieurs clubs).

**Résolution des doublons techniques**  
   Consolide les lignes strictement identiques par joueur, saison et club en fusionnant intelligemment les informations pour combler d éventuelles valeurs manquantes sans perte de données.

**Consolidation des transferts et recalcul des métriques**  
   Agrège les statistiques des joueurs ayant changé de club au cours d'une même saison (addition des buts, passes, minutes jouées, etc.) et réattribue leur dernier club en date. Enfin, les ratios statistiques (buts par 90 minutes, précision des tirs, temps de jeu moyen) sont recalculés à partir des totaux cumulés pour garantir des indicateurs exacts.

In [3]:
# Regardons tout d'abord s'il y a des doublons dans la base d'apprentissage

verifier_doublons_metier_et_techniques(df_base)

Recherche de doublons
 Aucune répétition technique (Même joueur, même saison, même club).

881 lignes correspondent à des doublons de mercato
(Même joueur, même saison, mais clubs différents : transferts de mi-saison)


{'doublons_techniques': np.int64(0), 'doublons_mercato': np.int64(881)}

In [4]:
# Ensuite, on fusionne les doublons techniques pour ne garder qu'une seule ligne par joueur

df_base = fusionner_doublons_techniques(df_base)

Format initial de la base : (16819, 121)
Format après fusion intelligente des doublons : (16819, 121)


In [5]:
# Enfin, on fusionne les doublons de mercato pour ne garder qu'une seule ligne par joueur par saison

df_base = fusionner_et_recalculer_mercato(df_base)

Format avant fusion mercato : (16819, 121)
Format après fusion mercato : (15938, 121)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)


### B2 : Homogénéisation des formats de date

Nous homogénéisons les formats de date entre les différentes bases afin d'être certains qu'elles soient comparables.

In [6]:
# On nettoie les colonnes de dates et on recalcule l'âge des joueurs
colonnes_dates = ["date_of_birth", "contract_expiration_date"]

df_base = nettoyer_age_et_dates(
    df_base, colonnes_dates=colonnes_dates
)

Traitement des colonnes de dates : ['date_of_birth', 'contract_expiration_date']
Toutes les heures ont été remises à minuit.
Calcul et nettoyage de la colonne 'age'...
0 âges manquants remplacés par la médiane (25 ans).
 -> Colonne 'age' convertie strictement en entiers (int).
Nettoyage de l'âge et des dates terminé.



### B3 : Traitement des variables présentant des valeurs manquantes

Les valeurs manquantes peuvent faire défaut dans notre future prédiction. C'est pourquoi il s'agit ici de les traiter.

Si le taux de valeurs manquantes dépasse un certain seuil, nous les imputons par des 0 ou la médiane selon le contexte.

In [7]:
diagnostiquer_valeurs_manquantes(df_base, seuil=0.03)

Diagnostic des valeurs manquantes (Seuil > 3%)
   Performance_PKwon : 100.0% de valeurs manquantes
   Performance_PKcon : 100.0% de valeurs manquantes
   Penalty Kicks_Save% : 94.7% de valeurs manquantes
   Performance_CS% : 92.8% de valeurs manquantes
   Performance_Save% : 92.6% de valeurs manquantes
   Performance_GA : 92.5% de valeurs manquantes
   Performance_GA90 : 92.5% de valeurs manquantes
   Performance_CS : 92.5% de valeurs manquantes
   Performance_D : 92.5% de valeurs manquantes
   Performance_W : 92.5% de valeurs manquantes
   Performance_Saves : 92.5% de valeurs manquantes
   Penalty Kicks_PKm : 92.5% de valeurs manquantes
   Penalty Kicks_PKatt : 92.5% de valeurs manquantes
   Penalty Kicks_PKsv : 92.5% de valeurs manquantes
   Penalty Kicks_PKA : 92.5% de valeurs manquantes
   Performance_L : 92.5% de valeurs manquantes
   Performance_SoTA : 92.5% de valeurs manquantes
   np_xg : 47.2% de valeurs manquantes
   xa : 47.2% de valeurs manquantes
   xg_buildup : 47.2% de v

In [8]:
# Application de la fonction
df_base = nettoyer_valeurs_manquantes_ciblees(df_base)

Début du traitement ciblé des valeurs manquantes...
 1525 lignes supprimées car 'market_value_in_eur' était manquant.
 -> 25 colonnes de performance nettoyées (NaN -> 0).


In [9]:
diagnostiquer_valeurs_manquantes(df_base, seuil=0.3)

Diagnostic des valeurs manquantes (Seuil > 30%)
   xg : 46.7% de valeurs manquantes
   xa : 46.7% de valeurs manquantes
   np_xg : 46.7% de valeurs manquantes
   xg_chain : 46.7% de valeurs manquantes
   xg_buildup : 46.7% de valeurs manquantes

Total : 5 colonnes dépassent le seuil de 30%.


### B4 Encodage de variables

Les variables catégorielles ne sont parfois pas inteprétables par certains modèles. Il s'agit donc ici de les transformer en variables numériques par un *One-Hot-Encoding* : une variable est créée pour chaque valeur.

Par exemple, la variable *foot* a 3 valeurs : `right`, `left` et `both`. 2 variables sont alors créées : droitier et gaucher. Si le joueur tire du pied droit, sa variable droitier vaudra 1 et sa variable gaucher 0. Pour un joueur ambidextre, ces 2 variables valent 1.

In [10]:
# Analyser la base
var_categorielles = lister_variables_categorielles(df_base)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   player (5117 modalités uniques)
   team (137 modalités uniques)
   league (5 modalités uniques)
   nation (125 modalités uniques)
   pos (10 modalités uniques)
   join_key (5116 modalités uniques)
   match_method (65 modalités uniques)
   sub_position (13 modalités uniques)
   position (5 modalités uniques)
   foot (3 modalités uniques)
   name (5117 modalités uniques)

Total : 11 variables catégorielles trouvées.


In [11]:
# Variables catégorielles à traiter
mes_variables = ["pos", "sub_position", "nation", "league", "foot"]

# Lancement de l'encodage
df_base = encoder_dataset(
    df=df_base,
    colonnes_categoriques=mes_variables,
    df_fifa_historique=df_fifa,
)

Format initial avant encodage : (14413, 121)
Encodage de 'nation' en 10 colonnes binaires Top FIFA (par saison)...
   • Variable 'est_anglais' créée.
   Variable 'confederation' encodée en colonnes binaires.
   Les 10 colonnes classement_FIFA_X ont été injectées.
Profil Joueurs de champ détecté : Encodage Multi-Label de 'pos'.
Encodage One-Hot des colonnes : ['sub_position', 'league', 'foot']
Format final après encodage : (14413, 160)



### B5 : Suppression de colonnes redondantes

Certaines variables étant fortement corrélées, il est inutile voire néfaste de toutes les conserver. En effet, certaines variables sont le produit ou l'addition de 2 autres, ce qui peut créer un certain bruit dans nos futurs modèles prédictifs.

C'est pourquoi nous supprimons certaines varibles, alors inutiles à la prédiction.

In [12]:
colonnes_redondantes = [
    "Starts_Starts", "Standard_PK", "Standard_PKatt", "Standard_Gls", "90s", "Playing Time_Min%",
    "Performance_SoTA", "Performance_G+A", "Team Success_+/-", "Team Success_+/-90", "Playing Time_Min", 
    "Penalty Kicks_PKatt", "born", "np_xg", "xg_chain", "Per 90 Minutes_G+A-PK", "Per 90 Minutes_G-PK",
    "join_key", "tm_join_key", "tm_join_key_full", "tm_id", "player_id", "dob_key", "tm_dob_key",
    "match_method", "name", "date_of_birth", "dob_year", "date", "season", "valuation_season_year",
    "contract_expiration_date", "Starts_Mn/Start", "Performance_W",
    "Performance_D", "Performance_L", "Performance_GA"
]

df_base = supprimer_colonnes_du_dataset(df_base, colonnes_redondantes)

30 colonne(s) supprimée(s) : ['Starts_Starts', 'Standard_PK', 'Standard_PKatt', 'Standard_Gls', '90s', 'Playing Time_Min%', 'Performance_SoTA', 'Performance_G+A', 'Team Success_+/-', 'Team Success_+/-90', 'Playing Time_Min', 'Penalty Kicks_PKatt', 'born', 'np_xg', 'xg_chain', 'Per 90 Minutes_G+A-PK', 'Per 90 Minutes_G-PK', 'join_key', 'tm_id', 'dob_key', 'match_method', 'name', 'date_of_birth', 'season', 'valuation_season_year', 'Starts_Mn/Start', 'Performance_W', 'Performance_D', 'Performance_L', 'Performance_GA']


### B6 : Sauvegarde de la pipeline de premier nettoyage

Nous sauvegardons la pipeline des étapes réalisées précédemment.

In [13]:
# Sauvegarde de la fonction dans le fichier .pkl

with open("../data_finale/pipelines/pipeline_nettoyage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_nettoyage, fichier)

### B7 : Traitement final

Cette étape exécute l'intégralité du traitement des données avant la modélisation afin de préparer des ensembles d'entraînement et d'évaluation propres, tout en garantissant **l'absence d'aide du futur (*data leakage*)**.


Le pipeline propose deux méthodes de séparation :
* **Temporel (par défaut) :** Découpage chronologique strict (Train : 2020-2022, Validation : 2023, Test : 2024 et Saison en cours : 2025).
* **Aléatoire :** Séparation classique basée sur un tirage aléatoire contrôlé par un *seed* (70% Train, 15% Validation, 15% Test).

**Traitement des valeurs aberrantes**
* Nettoyage de la taille des joueurs sur la plage définie (`155 cm` à `210 cm`).
* Remplacement des anomalies par la **médiane du poste** calculée uniquement sur l'ensemble d'entraînement.

**Imputation des métriques manquantes**
* Traitement des données manquantes en appliquant la **médiane croisée par Poste × Ligue**.
* Les médianes de référence sont apprises exclusivement sur l'ensemble Train pour éviter toute fuite d'information vers la validation et le test.

**Création de la variable de la valeur historique**
* Calcul de la valeur marchande logarithmique de la saison précédente pour chaque joueur via un décalage temporel.
* Système d'**imputation en cascade** à 4 niveaux pour combler les valeurs historiques manquantes (ex. nouveaux joueurs ou premières saisons) :
  1. Médiane par **Ligue × Poste × Saison**
  2. Médiane de secours par **Ligue × Poste**
  3. Médiane de secours par **Poste uniquement**
  4. Médiane **globale du Train**

**Exportation et sauvegarde**
* Export des quatre jeux de données nettoyés et imputés (`train.csv`, `val.csv`, `test.csv`, `en_cours.csv`) dans le dossier de destination spécifié.

In [14]:
df_train, df_val, df_test, df_en_cours = executer_pipeline_preprocessing(
    df = df_base,
    dossier_sortie=dossier_sortie,
    methode_split="temporel",
    height_min=155,
    height_max=210,
)

Split effectué, Train: 7404 | Val: 2431 | Test: 2295
Outliers traités (Bornes: [155, 210]. Remplacement par la médiane du poste du Train).
Imputation des NA terminée.
Colonne 'log_prev_value' calculée et imputée.
Pipeline terminé. Fichiers sauvegardés dans : ..\data_finale



Nous sauvegardons finalement cette seconde pipeline.

In [15]:
# Sauvegarde de la fonction dans le fichier .pkl

with open("../data_finale/pipelines/pipeline_anti_data_leakage.pkl", "wb") as fichier:
    pickle.dump(executer_pipeline_preprocessing, fichier)